<a href="https://colab.research.google.com/github/reavensullivan/Opinions/blob/main/Actividad4_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Configuración de GPU y Dependencias
import torch
print("¿GPU disponible?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Dispositivo:", torch.cuda.get_device_name(0))

# Instalar las librerías necesarias de manera silenciosa
!pip install -q transformers datasets scikit-learn pandas

¿GPU disponible?: False


In [ ]:
# 2. Carga de Librerías y Autenticación de Hugging Face Hub
import time
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Si requieres autenticarte en Hugging Face (opcional para modelos públicos)
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
# 3. Carga del Dataset
import pandas as pd
import urllib.request

try:
    print("Intentando cargar desde el repositorio Parquet de Hugging Face")
    # Cargamos el fragmento de validación del dataset oficial sst2 directamente desde sus splits estáticos de Hugging Face
    parquet_url = "https://huggingface.co/datasets/glue/resolve/main/sst2/validation-00000-of-00001.parquet"
    df = pd.read_parquet(parquet_url)
    # Tomar las primeras 300 muestras para optimizar velocidad y costos
    df = df.head(300).copy()
    df['label_text'] = df['label'].map({0: 'negative', 1: 'positive'})
    print("Dataset cargado con éxito.")
except Exception as e:
    print(f"Fallo la carga desde Parquet ({e}). Aplicando respaldo directo con cabecera de navegador")
    # Descarga directa del subset de validación de SST-2 con User-Agent modificado para evitar el HTTP Error 403
    url = "https://dl.fbaipublicfiles.com/glue/data/SST-2/dev.tsv"
    req = urllib.request.Request(
        url,
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}
    )
    with urllib.request.urlopen(req) as response:
        df_raw = pd.read_csv(response, sep='\t')
    df = df_raw.head(300).copy()
    df.rename(columns={'sentence': 'sentence', 'label': 'label'}, inplace=True)
    df['label_text'] = df['label'].map({0: 'negative', 1: 'positive'})
    print("Dataset cargado con éxito desde el servidor de respaldo de GLUE.")

display(df.head())

Intentando cargar desde el repositorio Parquet de Hugging Face
Dataset cargado con éxito.


,sentence,label,idx,label_text
0,it 's a charming and often affecting journey .,1,0,positive
1,unflinchingly bleak and desperate,0,1,negative
2,allows us to hope that nolan is poised to emba...,1,2,positive
3,"the acting , costumes , music , cinematography...",1,3,positive
4,"it 's slow -- very , very slow .",0,4,negative


In [ ]:
# 4. Definición de Modelos y Evaluación
model_names = {
    "DistilBERT": "distilbert-base-uncased-finetuned-sst-2-english",
    "RoBERTa": "siebert/sentiment-roberta-large-english",
    "MobileBERT": "Alireza1044/mobilebert_sst2"
}

results = []
device = 0 if torch.cuda.is_available() else -1

for name, path in model_names.items():
    print(f"Evaluando {name}...")

    # Inicializar el pipeline
    classifier = pipeline("sentiment-analysis", model=path, device=device)

    predictions = []
    latencies = []

    # Medir inferencia por registro
    for text in df['sentence']:
        start_time = time.time()
        pred = classifier(text)[0]
        end_time = time.time()

        latencies.append(end_time - start_time)

        # Normalizar las etiquetas del output del modelo
        pred_label = pred['label'].lower()
        if 'pos' in pred_label or '1' in pred_label or 'label_1' in pred_label:
            predictions.append(1)
        else:
            predictions.append(0)

    # Métricas globales
    acc = accuracy_score(df['label'], predictions)
    prec, rec, f1, _ = precision_recall_fscore_support(df['label'], predictions, average='binary')
    avg_latency = np.mean(latencies) * 1000 # en milisegundos

    results.append({
        "Modelo": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "Latencia Promedio (ms)": round(avg_latency, 2)
    })

Evaluando DistilBERT...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

model.safetensors: downloading bytes:           |  0.00B            

tokenizer_config.json:   0%|          | 0.00/335 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [ ]:
# 5. Tabla Comparativa de Resultados
df_results = pd.DataFrame(results)
display(df_results)

,Modelo,Accuracy,Precision,Recall,F1-Score,Latencia Promedio (ms)
0,DistilBERT,0.9067,0.8944,0.9290,0.9114,175.74
1,RoBERTa,0.9167,0.9062,0.9355,0.9206,632.05
2,MobileBERT,0.8933,0.8868,0.9097,0.8981,61.28


```markdown
## 6. Exportación de Resultados y Dataset de Soporte

Para asegurar la reproducibilidad requerida en las instrucciones, guardaremos de manera local las métricas comparativas y el subset del dataset evaluado en los archivos CSV correspondientes.
```

In [ ]:
# 1. Guardar la tabla de métricas comparativas
df_results.to_csv('/content/metricas_comparativas.csv', index=False)
print("Archivo 'metricas_comparativas.csv' guardado exitosamente.")

# 2. Guardar el dataset con los textos y etiquetas de referencia para soporte
df[['sentence', 'label', 'label_text']].to_csv('/content/sst2_dataset_eval.csv', index=False)
print("Archivo 'sst2_dataset_eval.csv' guardado exitosamente.")

Archivo 'metricas_comparativas.csv' guardado exitosamente.
Archivo 'sst2_dataset_eval.csv' guardado exitosamente.
